In [ ]:
import math
import json
from utils import btree

In [ ]:
def load_char_to_matrix_pairs(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return list(data["glyphs"].items())

In [ ]:
def compute_pixel_heatmap(ctm_pairs):
    if len(ctm_pairs) == 0:
        raise ValueError("empty ctm pairs")
    # we'll use  the first matrix to define dimensions (intentional hack)
    _, first_matrix = ctm_pairs[0]
    height = len(first_matrix)
    width = len(first_matrix[0])

    # initializing heatmap
    heatmap = [[0 for _ in range(width)] for _ in range(height)]

    # count how many matrices (i.e., glyphs) have a 1 at each (y, x)
    for _, matrix in ctm_pairs:
        for y in range(height):
            for x in range(width):
                if matrix[y][x] == 1:
                    heatmap[y][x] += 1

    return heatmap

In [ ]:
def identify_discriminative_pixels(heatmap, charset_length, max_depth, curr_depth):
    threshold = int(math.pow(2, max_depth - curr_depth - 1))
    # print(f"threshold at depth {curr_depth} = {threshold}")
    dpxls = []
    height = len(heatmap)
    width = len(heatmap[0])
    for y in range(height):
        for x in range(width):
            count = heatmap[y][x]
            normalized_count = max(count, charset_length - count)
            if normalized_count != charset_length and normalized_count <= threshold:
                dpxls.append((x, y))
    # note: this line is optional. I am simply adding it for the assumption that it'll make things converge faster. However, I did not bother to verify that.
    dpxls.sort(key=lambda p: abs(heatmap[p[1]][p[0]] - (charset_length / 2)))
    return dpxls

In [ ]:
def split_pairs_by_discriminative_pixel(ctm_pairs, dpixel_x, dpixel_y):
    left_pairs = []
    right_pairs = []
    for p in ctm_pairs:
        _, matrix = p
        if matrix[dpixel_y][dpixel_x] == 1:
            left_pairs.append(p)
        else:
            right_pairs.append(p)
    return left_pairs, right_pairs

In [ ]:
def build_ocr_btree(ctm_pairs, max_depth, curr_depth=0):
    # check if we can't go deeper
    if curr_depth > max_depth:
        print("reached max depth")
        return False, None

    if len(ctm_pairs) == 0:
        raise ValueError("empty ctm pairs")

    if len(ctm_pairs) == 1:
        char = ctm_pairs[0][0]
        # print(f"leaf found {char}")
        return True, btree.leaf(char)

    # compute heatmap
    heatmap = compute_pixel_heatmap(ctm_pairs)
    charset_len = len(ctm_pairs)
    # identify discriminative pixels
    dpixels = identify_discriminative_pixels(
        heatmap=heatmap,
        charset_length=charset_len,
        max_depth=max_depth,
        curr_depth=curr_depth
    )

    # check if discriminative pixels is empty (return)
    if len(dpixels) == 0:
        # print("no discriminative pixel found")
        return False, None

    # iterating (depth first ) over all discriminative pixels
    for x, y in dpixels:
        # print(f"Attempting split by: {x}, {y}")
        lpairs, rpairs = split_pairs_by_discriminative_pixel(ctm_pairs, x, y)
        # print(f"split: len(lpairs) = {len(lpairs)} and len(rpairs) = {len(rpairs)}")

        is_left_successful, ltree = build_ocr_btree(lpairs, max_depth, curr_depth + 1)
        if not is_left_successful:
            # try another discriminative pixel
            continue

        is_right_successful, rtree = build_ocr_btree(rpairs, max_depth, curr_depth + 1)

        if not is_right_successful:
            # try another discriminative pixel
            continue

        # print("tree found")
        return True, btree.Node((x, y), ltree, rtree)

    # If we reach this, and nothing was found, then we're doomed
    # print("could not build tree within the given max depth")
    return False, None

In [ ]:
# loading our char_to_matrix pairs
char_to_matrix_pairs = load_char_to_matrix_pairs("../out/charset_glyphs.json")
number_of_chars = len(char_to_matrix_pairs)
print(f"total number of characters: {number_of_chars}")

# calculating the ideal depth of our btree
# max_depth = 6 (given by the game)
ideal_max_depth = math.ceil(math.log2(number_of_chars))
print("ideal max depth:", ideal_max_depth)

# building our pixel perfect ocr btree
is_successful, ocr_btree = build_ocr_btree(char_to_matrix_pairs, max_depth=ideal_max_depth)

# checking everything went well
print(f"Did we successfully build our btree? {is_successful}")
if is_successful:
    # uncomment this to print btree
    # btree.print_btree(ocr_btree)

    # exporting btree
    out_ocr_btree_path = "../out/ocr_btree.json"
    print(f"Saving our btree to {out_ocr_btree_path}")
    btree.save_btree_json(ocr_btree, out_ocr_btree_path)

    # loading btree (just to doublecheck)
    print(f"Loading our btree from {out_ocr_btree_path}")
    loaded_ocr_btree = btree.load_btree_json(out_ocr_btree_path)
    btree.print_btree(loaded_ocr_btree)